# Du champ à la nappe : glyphosate et AMPA

**La question** : dans un département, est-ce qu'on retrouve dans les eaux souterraines ce qui y est vendu ? Et quand les ventes baissent, est-ce que les nappes suivent ?

**Le trajet**

| Partie | Ce que tu fais | Ce que tu travailles |
|---|---|---|
| A | Récupérer les ventes de glyphosate du département, par année | l'API que tu connais, un filtre de territoire |
| B | Récupérer les analyses de glyphosate et d'AMPA dans les nappes | **une nouvelle API**, la pagination, la limite des 20 000 lignes |
| C | Comprendre et nettoyer les analyses | lire des codes, dates, `to_datetime`, `.dt.year` |
| D | Résumer les analyses par année | colonnes de `True`/`False`, `groupby`, `agg` |
| E | Relier ventes et analyses | **`merge`** |
| F | Graphique et conclusion | `secondary_y`, ta fiche graphiques |

**Organisation, comme les pros**
- `data/` : chaque tableau récupéré est enregistré en CSV dès qu'il est propre. Tu ne refais plus les requêtes : tu relis le fichier.
- `figures/` : chaque graphique final est enregistré en PNG.
- Une cellule **Constat** est prévue sous chaque résultat important : une ou deux phrases, avec le chiffre. Remplace les `…`.
- `README.md` : à remplir à la fin, en 10 lignes.

**Règles du jeu** : indices repliés, le 1 avant le 2, pas de solution dans le fichier. Après deux essais, envoie-moi ton code.

**Ce que je n'ai pas pu vérifier** : je n'ai pas accès à ces API depuis chez moi. Les adresses et les noms de paramètres viennent de leur documentation, mais je ne connais ni les noms exacts des colonnes des analyses, ni le nombre de lignes. Les parties B et C te les font découvrir.

In [29]:
%load_ext autoreload
%autoreload 2
import requests
import pandas as pd
import matplotlib.pyplot as plt
from src.hubeau import recuperer_toutes_les_pages

DEPARTEMENT = "24"      # change ici si le département a trop peu d'analyses
CODE_GLYPHOSATE = 1506  # code SANDRE, le même dans les deux bases
CODE_AMPA = 1907        # code SANDRE de l'AMPA, le métabolite du glyphosate

URL_VENTES = "https://hubeau.eaufrance.fr/api/v1/vente_achat_phyto/ventes/substances"
URL_NAPPES = "https://hubeau.eaufrance.fr/api/v1/qualite_nappes/analyses"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Partie A. Les ventes du département

### A1. La requête

**Ce que tu as** : l'adresse des ventes, et ce que dit la documentation : pour un département, `type_territoire` vaut `"Département"` (avec l'accent) et `code_territoire` vaut le numéro du département. On peut filtrer la substance par son code avec `code_substance`, plus sûr que le nom (souviens-toi des oxychlorures).

**Ce que tu veux** : `ventes`, un DataFrame de toutes les ventes de glyphosate du département, toutes années disponibles. Affiche le code de retour, `count`, et `next`.

Si le code de retour n'est pas 200 ou si `count` vaut 0, ne t'acharne pas : essaie `"Departement"` sans accent, et dis-moi ce qui a marché.


<details><summary>Indice 1</summary>

C'est la requête de ton projet cuivre, avec d'autres paramètres. Pas besoin d'`annee_min` : sans, tu as toutes les années.

</details>

<details><summary>Indice 2, plus précis</summary>

Les paramètres : `code_substance`, `type_territoire`, `code_territoire`, `size`. Puis `pd.DataFrame(reponse["data"])`.

</details>

In [30]:
parametres={
    "code_substance":CODE_GLYPHOSATE,
    "type_territoire":"Département",
    "code_territoire":24
    }

In [32]:
donnees.status_code

200

In [34]:
type(vente_api),vente_api.keys()

(dict,
 dict_keys(['count', 'first', 'last', 'prev', 'next', 'api_version', 'data']))

In [35]:
len(vente_api["data"])

1305

In [37]:
vente_api["count"]

1305

In [39]:
vente_api

,amm,annee,classification,classification_mention,code_cas,code_substance,code_territoire,fonction,libelle_substance,quantite,libelle_territoire,type_territoire,uri_substance,uri_territoire
0,2120155,2024,Env A,NaN,1071-83-6,1506,24,Herbicide,glyphosate,6.600,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
1,2100006,2024,Env A,NaN,1071-83-6,1506,24,Herbicide,glyphosate,1.080,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
2,2130253,2024,Env A,NaN,1071-83-6,1506,24,Herbicide,glyphosate,108.120,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
3,2090093,2024,Env A,NaN,1071-83-6,1506,24,Herbicide,glyphosate,9981.000,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
4,2150140,2024,Env A,NaN,1071-83-6,1506,24,Herbicide,glyphosate,52093.800,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1300,2070053,2008,N Organique,-,1071-83-6,1506,24,Herbicide,glyphosate,57.600,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
1301,9500427,2008,N Organique,-,1071-83-6,1506,24,Herbicide,glyphosate,216.000,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
1302,9500449,2008,N Organique,-,1071-83-6,1506,24,Herbicide,glyphosate,312.375,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
1303,2000305,2008,N Organique,-,1071-83-6,1506,24,Herbicide,glyphosate,1101.600,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24


In [40]:
len(vente_api)

1305

In [41]:
vente_api.count()

amm                       1305
annee                     1305
classification            1305
classification_mention    1169
code_cas                  1305
code_substance            1305
code_territoire           1305
fonction                  1305
libelle_substance         1305
quantite                  1305
libelle_territoire        1305
type_territoire           1305
uri_substance             1305
uri_territoire            1305
dtype: int64

Avec utilisation de fichiers .py, requirements...

In [42]:
lignes = recuperer_toutes_les_pages(URL_VENTES, parametres)

In [43]:
vente_api = pd.DataFrame(lignes)

In [44]:
len(vente_api)

1305

### A2. Vérifier, puis enregistrer

**Ce que tu veux** :
1. vérifier que toutes les lignes concernent bien ton département et le glyphosate (`value_counts` sur les bonnes colonnes) ;
2. regarder les années couvertes ;
3. vérifier les doublons ;
4. enregistrer le tableau dans `data/ventes_glyphosate.csv`, sans l'index.

In [45]:
vente_api["libelle_substance"].value_counts()

libelle_substance
glyphosate                   1297
n-phosphonomethyl glycine       8
Name: count, dtype: int64

In [46]:
vente_api["code_territoire"].value_counts()

code_territoire
24    1305
Name: count, dtype: int64

In [47]:
vente_par_annee=vente_api.groupby("annee")["quantite"].sum()

DOUBLONS

In [48]:
vente_api.duplicated().sum()

np.int64(0)

In [49]:
vente_api[vente_api.duplicated(subset=["amm","annee"], keep=False)]

,amm,annee,classification,classification_mention,code_cas,code_substance,code_territoire,fonction,libelle_substance,quantite,libelle_territoire,type_territoire,uri_substance,uri_territoire
1079,2090044,2010,N Organique,-,1071-83-6,1506,24,Herbicide,glyphosate,3.4822,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24
1094,2090044,2010,N Organique,-,1071-83-6,1506,24,Herbicide,n-phosphonomethyl glycine,3.4822,DORDOGNE,Département,http://id.eaufrance.fr/par/1506,http://id.eaufrance.fr/dep/24


suppression du doublon: 2x 3,4822 kg en 2010 pour l'AMM 2090044. ==> une ligne pour glyphosate une ligne pour n-phosphonomethyl glycine

In [50]:
vente_api_propre=vente_api.drop_duplicates(subset=["amm", "annee","quantite"])

In [51]:
vente_api_propre["code_territoire"].value_counts()

code_territoire
24    1304
Name: count, dtype: int64

> **Constat** : années couvertes, nombre de lignes, doublons éventuels.
> - Bien de filtrer par nom de code_substance car noms différents pour le glyphosate!

> Jeux de données: 2008 à 2024.


### A3. Les ventes par année

**Ce que tu veux** : `ventes_par_annee`, une Series : la quantité totale vendue chaque année, **en kg**. Et la même chose en nombre de produits différents.

> **Constat** : tendance des ventes, année la plus haute, année la plus basse.

## Partie B. Les analyses dans les nappes

### B1. Une première page, pour découvrir

**Ce que tu as** : l'adresse des analyses. D'après la documentation, on filtre le département avec `num_departement` et la substance avec `code_param`, qui accepte plusieurs codes séparés par une virgule.

**Ce que tu veux** : une petite requête, `size` à 20, pour le glyphosate **et** l'AMPA dans ton département. Affiche le code de retour, `count`, puis **les clés de la première analyse**.

Lis bien ces clés : tu vas devoir y trouver toi-même la date, la valeur mesurée, l'unité, la substance et le point de mesure. Note-les dans le constat.


<details><summary>Indice 1</summary>

Tu as déjà fait cette découverte avec la première réponse des ventes : clés du dictionnaire, puis clés de la première ligne de `data`.

</details>

<details><summary>Indice 2, plus précis</summary>

`code_param` peut recevoir le texte `f"{CODE_GLYPHOSATE},{CODE_AMPA}"`. Les clés d'un dictionnaire : `.keys()`.

</details>

> **Constat** : nombre total d'analyses annoncé ; noms des colonnes utiles (date, résultat, unité, substance, station).

### B2. Tout récupérer

**Ce que tu veux** : `analyses`, un DataFrame de toutes les analyses de glyphosate et d'AMPA du département.

Deux contraintes de cette API, données par sa documentation :
- **20 000 lignes au maximum par page** ;
- **20 000 lignes au maximum en tout**, page × taille. Au-delà, l'API refuse d'aller plus loin.

Regarde ton `count` de B1 et choisis ta stratégie :
- **moins de 20 000** : la boucle de pages de ton projet cuivre suffit ;
- **plus de 20 000** : il faut découper la demande, par exemple **une requête par substance**, ou par tranches de dates avec `date_debut_prelevement` et `date_fin_prelevement`. Chaque morceau doit faire moins de 20 000 lignes.

Puis vérifie que `len(analyses)` est égal au total annoncé.


<details><summary>Indice 1</summary>

Liste vide avant la boucle, `extend` à chaque page, arrêt quand `next` vaut `None`. Comme en A2 du cuivre.

</details>

<details><summary>Indice 2, plus précis</summary>

Avec `size` à 5000, quatre pages suffisent pour 20 000 lignes. Pense à recréer les paramètres à chaque tour si tu boucles sur les substances.

</details>

### B3. Enregistrer, puis relire

**Ce que tu veux** : enregistrer `analyses` dans `data/analyses_nappes.csv`, puis le **relire** avec `pd.read_csv` dans un nouveau tableau et comparer les deux : même nombre de lignes ? mêmes types de colonnes (`dtypes`) ?

Il y a souvent une surprise à la relecture : un code comme `"0123"` relu en nombre perd son zéro. Si c'est le cas pour une colonne, note-le.

À partir d'ici, tu peux travailler sur le fichier sans refaire les requêtes.

> **Constat** : taille du fichier, éventuels changements de type à la relecture.

## Partie C. Comprendre et nettoyer les analyses

### C1. Inventaire

**Ce que tu veux**, colonne par colonne, avec `value_counts` ou `isna` :
1. combien d'analyses pour le glyphosate, combien pour l'AMPA ;
2. **quelle unité** : tout est-il dans la même ? Le seuil de 0,1 µg/L n'a de sens que dans cette unité ;
3. combien de points de mesure différents ;
4. les valeurs manquantes dans la colonne du résultat.

> **Constat** : …

### C2. La colonne de remarque : quantifié ou pas ?

Dans les analyses d'eau, beaucoup de résultats ne sont pas de vraies mesures : la substance était **sous la limite de quantification**, et le laboratoire écrit alors la valeur de cette limite à la place du résultat. Une colonne de **remarque** dit lequel des deux cas s'applique.

**Ce que tu veux** : trouver cette colonne parmi les clés de B1, afficher ses valeurs avec `value_counts()`, et comprendre ce que veut dire chaque code.

Dans les référentiels SANDRE, le code **1** correspond en principe à un résultat quantifié, et le code **10** à un résultat inférieur à la limite de quantification. Je n'ai pas pu le vérifier sur ces données : regarde si la colonne porte aussi un libellé en clair, et fie-toi à lui.

Puis crée une colonne booléenne `quantifie`.


<details><summary>Indice 1</summary>

C'est une colonne de `True`/`False`, comme `alerte` ou `est_cuivre` : une comparaison sur une colonne.

</details>

<details><summary>Indice 2, plus précis</summary>

`analyses["quantifie"] = analyses["…"] == …`, avec la colonne et le code que tu as trouvés.

</details>

> **Constat** : part des analyses réellement quantifiées.

### C3. Les dates

**Ce que tu as** : une colonne de date de prélèvement, en texte.

**Ce que tu veux** : une colonne `date` de vraies dates, avec `pd.to_datetime`, puis une colonne `annee` qui contient l'année de chaque date.

Pour la deuxième, la même idée que `self.date.year` dans ta classe `Observation`, mais pour toute une colonne d'un coup : pandas passe par `.dt`, puis le même attribut.

Contrôle : les années couvertes, avec `value_counts().sort_index()`. Y a-t-il des années très anciennes, ou bizarres ?


<details><summary>Indice 1</summary>

`pd.to_datetime(colonne)` transforme une colonne de textes en colonne de dates. Si certaines dates sont impossibles, `errors="coerce"` les remplace par des valeurs manquantes au lieu de planter.

</details>

<details><summary>Indice 2, plus précis</summary>

`analyses["annee"] = analyses["date"].dt.year`

</details>

> **Constat** : période couverte par les analyses, comparée à celle des ventes.

### C4. Le seuil

**Ce que tu veux** : une colonne booléenne `depasse`, vraie quand l'analyse est **quantifiée** et que le résultat est **strictement supérieur à 0,1** µg/L.

Deux conditions à la fois sur des colonnes pandas : on ne les relie pas avec `and`, mais avec `&`, et chaque condition entre parenthèses.

> **Constat** : nombre et part des analyses au-dessus du seuil, pour chaque substance.

## Partie D. Résumer les analyses par année

### D1. Une ligne par année et par substance

**Ce que tu veux** : `resume`, un tableau avec une ligne par (année, substance) et quatre colonnes :
- le nombre d'analyses ;
- le nombre d'analyses quantifiées ;
- la part d'analyses au-dessus du seuil ;
- le nombre de points de mesure différents.

Un seul `groupby` sur deux colonnes, puis `agg`. Pour avoir des colonnes avec des noms clairs, pandas accepte cette forme :

```python
tableau.groupby(["a", "b"]).agg(
    nom_colonne_1=("colonne", "calcul"),
    nom_colonne_2=("autre_colonne", "autre_calcul"),
)
```

À toi de choisir, pour chaque ligne, la colonne et le calcul : `count`, `sum`, `mean`, `nunique`. Rappel : la moyenne d'une colonne de `True`/`False` est une proportion.

### D2. Attention au nombre d'analyses

Le nombre d'analyses peut beaucoup varier d'une année à l'autre, selon les campagnes de mesure. **Une année avec plus de dépassements n'est pas forcément une année plus polluée** : c'est peut-être juste une année avec plus de prélèvements.

**Ce que tu veux** : afficher le nombre d'analyses par année et repérer les années trop pauvres pour qu'on en tire une conclusion. Décide d'un seuil, par exemple « au moins 30 analyses », et note-le.

> **Constat** : années retenues pour la comparaison, et pourquoi.

## Partie E. Relier ventes et analyses

### E1. Préparer les deux tableaux

Pour relier deux tableaux, il faut une colonne commune, avec **le même nom et le même type** des deux côtés.

**Ce que tu veux** :
- `ventes_annuelles` : un DataFrame avec deux colonnes, `annee` et `kg_vendus` ;
- `glypho_annuel` et `ampa_annuel` : deux DataFrame tirés de `resume`, une ligne par année, avec la part au-dessus du seuil.

Si ta Series a l'année en index, `.reset_index()` la transforme en colonne.

### E2. Le merge

`pd.merge` relie deux tableaux sur une colonne commune, comme une RECHERCHEV qui ramènerait toutes les colonnes d'un coup :

```python
pd.merge(tableau_gauche, tableau_droite, on="colonne_commune", how="inner")
```

Le paramètre `how` décide quoi faire des années présentes d'un seul côté :

| `how` | Années gardées |
|---|---|
| `"inner"` | seulement celles présentes **des deux côtés** |
| `"left"` | toutes celles du tableau de gauche |
| `"outer"` | toutes, des deux côtés ; les trous deviennent des valeurs manquantes |

**Ce que tu veux** : `relie`, les ventes et les deux substances côte à côte, une ligne par année. Essaie `inner` puis `outer`, et compare le nombre de lignes : qu'est-ce que ça t'apprend sur les périodes couvertes ?

Si deux colonnes portent le même nom des deux côtés, pandas ajoute `_x` et `_y`. Le paramètre `suffixes=("_glypho", "_ampa")` te laisse choisir.

> **Constat** : nombre d'années communes ; ce que `outer` ajoute.

## Partie F. Graphique et conclusion

### F1. Un graphique à deux axes

Ventes et part de dépassements n'ont pas la même unité : des kilos et des pourcentages. Sur le même axe, une des deux courbes serait écrasée.

**Ce que tu veux** : un seul graphique, ventes en barres ou en courbe sur l'axe de gauche, part de dépassements pour le glyphosate et l'AMPA sur l'axe de droite. Avec titre, noms des axes, légende lisible. Enregistre-le dans `figures/`.

Ta fiche graphiques a ce qu'il faut : `secondary_y`.

### F2. Le décalage

Les molécules mettent du temps à rejoindre les nappes et à en disparaître. L'AMPA, en particulier, persiste plus longtemps que le glyphosate.

**Ce que tu veux** : regarder ton graphique et ton tableau `relie`, puis répondre :
- quand les ventes baissent, la part de dépassements baisse-t-elle la même année, ou plus tard ?
- l'AMPA et le glyphosate se comportent-ils pareil ?

Pour aller plus loin : la méthode `.shift(1)` décale une colonne d'une ligne. Une colonne « ventes de l'année précédente » te permet de comparer chaque année aux ventes de l'année d'avant.

### F3. Ta conclusion

- **Les ventes** :
- **Les nappes** :
- **Le lien, ou son absence** :
- **Les limites** : ventes déclarées par le lieu du distributeur et non d'usage ; points de mesure qui changent d'une année à l'autre ; limite de quantification qui baisse avec le temps et fait apparaître des détections ; une corrélation n'est pas une cause.

### F4. Le README

Ouvre `README.md` dans le dossier du projet et remplis-le : c'est la page qu'un recruteur lira en premier sur ton GitHub.